# 04 ? Build a Vector Database Workflow with Qdrant

This is the first hands-on vector database notebook. We will build a small searchable help-center collection with Qdrant and follow records through creation, query, retrieval, metadata updates, vector updates, deletion, and a local persistence example.

We will use the Qdrant Python client with its local FastEmbed integration. This keeps the lab self-contained: Qdrant runs in the Python process for this small dataset, while FastEmbed creates actual text embeddings. A local in-memory client is for learning and small experiments; it is not a production deployment. The later production notebook will cover service architecture, backups, and access control.

Official references: [Qdrant FastEmbed guide](https://qdrant.tech/documentation/fastembed/fastembed-semantic-search/), [points and CRUD](https://qdrant.tech/documentation/manage-data/points/), [collections](https://qdrant.tech/documentation/manage-data/collections/), and [payload indexes](https://qdrant.tech/documentation/concepts/payload/).

## Learning goals

By the end, you should be able to:

1. Start a local Qdrant client and create a collection with a vector size and metric.
2. Represent a chunk as a Qdrant point with a supported ID, vector, and payload.
3. Upsert points and run semantic top-K queries.
4. Retrieve by ID, scroll through records, and filter search with payload fields.
5. Patch payload data, replace an updated text/vector record, and delete records.
6. Explain the difference between in-memory mode, local persistent mode, and a Qdrant server.

The Qdrant terms used here are **collection**, **point**, and **payload**. The point is the stored record; its ID identifies it, its vector supports similarity search, and its payload carries associated fields.

## 1. Install the Python client and local embedding support

The optional `fastembed` extra lets the Python client embed text locally using `models.Document`. The first use of the embedding model may download model files, so the first run takes longer and needs network access. The model is used to make this notebook's search meaningful; in your own RAG pipeline, you can use the embedding model you selected earlier and pass its vectors to Qdrant instead.

Run this installation cell in the notebook's active Python environment, then restart the kernel if the notebook environment asks you to.

In [1]:
%pip install "qdrant-client[fastembed]>=1.14.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 9.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 52.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.1/324.1 kB 18.2 MB/s eta 0:00:00


The official examples use the FastEmbed extra and a local `QdrantClient(":memory:")`. This notebook follows that API shape. If Qdrant's client changes after this notebook is written, compare with the official docs linked above.

In [2]:
from uuid import NAMESPACE_URL, uuid5

from qdrant_client import QdrantClient, models

MODEL_NAME = "BAAI/bge-small-en"
COLLECTION_NAME = "rag_scratch_qdrant_lab"

client = QdrantClient(":memory:")
vector_size = client.get_embedding_size(MODEL_NAME)
print("Embedding model:", MODEL_NAME)
print("Vector size:", vector_size)
print("Collection:", COLLECTION_NAME)

Embedding model: BAAI/bge-small-en
Vector size: 384
Collection: rag_scratch_qdrant_lab


### Why this is a local lab

`QdrantClient(":memory:")` runs local storage in the current Python process. It avoids Docker and a separate server, but data disappears when the client/kernel is closed. It is useful for learning point and query operations with a small dataset. It does not demonstrate a production service, network access, multi-user concurrency, or a backup/restore plan.

## 2. Prepare a tiny RAG-style corpus

Each dictionary below represents a chunk that might come from a help-center article. We will store the text in the payload so the search result is directly useful to a RAG application. The remaining payload fields support source display, filtering, and maintenance.

Qdrant point IDs must be unsigned integers or UUIDs. Here we derive a deterministic UUID from the document ID and chunk number. This keeps repeated ingestion of the same logical chunk mapped to the same point ID, as long as the chunking scheme remains stable.

In [3]:
documents = [
    {
        "document_id": "account-access",
        "chunk_index": 0,
        "tenant_id": "acme",
        "category": "account",
        "source": "employee-handbook",
        "text": "To sign in, use your work email and the password associated with your account.",
    },
    {
        "document_id": "account-access",
        "chunk_index": 1,
        "tenant_id": "acme",
        "category": "account",
        "source": "employee-handbook",
        "text": "If you forget your password, choose Forgot password on the sign-in page and follow the email instructions.",
    },
    {
        "document_id": "account-access",
        "chunk_index": 2,
        "tenant_id": "acme",
        "category": "account",
        "source": "employee-handbook",
        "text": "Multi-factor authentication adds a second verification step after you enter your password.",
    },
    {
        "document_id": "expense-policy",
        "chunk_index": 0,
        "tenant_id": "acme",
        "category": "finance",
        "source": "expense-policy-2026",
        "text": "Submit business travel expenses within thirty days and attach itemized receipts.",
    },
    {
        "document_id": "expense-policy",
        "chunk_index": 1,
        "tenant_id": "acme",
        "category": "finance",
        "source": "expense-policy-2026",
        "text": "A manager must approve an expense report before reimbursement is issued.",
    },
    {
        "document_id": "device-support",
        "chunk_index": 0,
        "tenant_id": "acme",
        "category": "it-support",
        "source": "it-guide",
        "text": "Restart your laptop if it becomes unresponsive. Save your work first when possible.",
    },
    {
        "document_id": "device-support",
        "chunk_index": 1,
        "tenant_id": "acme",
        "category": "it-support",
        "source": "it-guide",
        "text": "Connect to the company VPN before opening internal tools from an off-site network.",
    },
    {
        "document_id": "travel-policy",
        "chunk_index": 0,
        "tenant_id": "globex",
        "category": "travel",
        "source": "travel-policy",
        "text": "Book economy airfare for domestic business trips unless an approved exception applies.",
    },
    {
        "document_id": "travel-policy",
        "chunk_index": 1,
        "tenant_id": "globex",
        "category": "travel",
        "source": "travel-policy",
        "text": "Keep hotel receipts and include the business purpose when submitting travel expenses.",
    },
]

point_ids = [
    str(uuid5(NAMESPACE_URL, f"{doc['document_id']}:{doc['chunk_index']}"))
    for doc in documents
]
payloads = [doc.copy() for doc in documents]

list(zip(point_ids[:3], [doc["text"] for doc in documents[:3]]))

[('86e06dae-8c6e-58ce-963c-a362c436b89b',
  'To sign in, use your work email and the password associated with your account.'),
 ('9caffaf4-a347-5565-a73f-67a4107cb87d',
  'If you forget your password, choose Forgot password on the sign-in page and follow the email instructions.'),
 ('9f26bab8-eddf-555f-aa96-6af9557a5301',
  'Multi-factor authentication adds a second verification step after you enter your password.')]

The two tenants are deliberate. Later we will query only `acme` records. This is a teaching example for metadata filters, not a security boundary: an application must enforce authorization correctly, and production deployments need suitable access controls.

## 3. Create a collection

A Qdrant collection groups points that share vector configuration. We set the vector size from the embedding model instead of hardcoding it, then use cosine distance for this model. The collection configuration must agree with the vectors you insert and queries you send.

This cell removes an existing collection with this lab-only name so you can rerun the notebook from the top. Do not use this reset pattern against a collection containing data you need.

In [4]:
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=vector_size,
        distance=models.Distance.COSINE,
    ),
)

collection_info = client.get_collection(COLLECTION_NAME)
collection_info

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=0, segments_count=1, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optim

### What did collection creation decide?

- `size` fixes the vector dimension expected by the collection.
- `distance` tells Qdrant how to score vector similarity.
- The collection name is a logical grouping; it is not itself a tenant access-control mechanism.

We chose cosine because it is a common choice for text embeddings. Always follow the embedding model's intended similarity setup. A collection cannot make vectors from incompatible embedding spaces comparable just because their dimensions match.

## 4. Upsert points with text embeddings

An upsert inserts a new point or overwrites a point with the same ID. Here the client embeds each text locally through FastEmbed and sends the resulting vectors to the local Qdrant store. The payload stays attached to the same point.

Stable IDs make repeated uploads easier to reason about: rerunning the same upsert does not create a second logical chunk with a different ID. If chunk boundaries change, ingestion still needs to remove old IDs that are no longer part of the document.

In [5]:
client.upload_collection(
    collection_name=COLLECTION_NAME,
    vectors=[
        models.Document(text=doc["text"], model=MODEL_NAME)
        for doc in documents
    ],
    payload=payloads,
    ids=point_ids,
)

count = client.count(collection_name=COLLECTION_NAME, exact=True)
print("Points stored:", count.count)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Points stored: 9


## 5. Run a semantic top-K query

The query text is embedded with the same model and searched against the collection. Qdrant returns points ordered by relevance for this metric. We request payloads so we can inspect the matched text and source. We do not request the vector itself because this application flow does not need to display it.

In [6]:
query_text = "How can I reset my account password?"

result = client.query_points(
    collection_name=COLLECTION_NAME,
    query=models.Document(text=query_text, model=MODEL_NAME),
    limit=4,
    with_payload=True,
    with_vectors=False,
)

for rank, hit in enumerate(result.points, start=1):
    print(f"{rank}. score={hit.score:.4f} id={hit.id}")
    print("   text:", hit.payload["text"])
    print("   source:", hit.payload["source"])
    print()

1. score=0.8823 id=9caffaf4-a347-5565-a73f-67a4107cb87d
   text: If you forget your password, choose Forgot password on the sign-in page and follow the email instructions.
   source: employee-handbook

2. score=0.8497 id=86e06dae-8c6e-58ce-963c-a362c436b89b
   text: To sign in, use your work email and the password associated with your account.
   source: employee-handbook

3. score=0.8253 id=9f26bab8-eddf-555f-aa96-6af9557a5301
   text: Multi-factor authentication adds a second verification step after you enter your password.
   source: employee-handbook

4. score=0.8251 id=26ae8f19-6866-5167-8c39-80cbe94d16e2
   text: Restart your laptop if it becomes unresponsive. Save your work first when possible.
   source: it-guide



### Read the result carefully

- **Top-K** is controlled by `limit`; it asks Qdrant for at most that many matches.
- **Score** is meaningful only together with the metric and embedding model. For cosine similarity, larger scores indicate more similar results in this query response.
- **Payload** carries the text and source back to the application. It can be omitted or narrowed when a request does not need all fields.
- A high similarity score is not a calibrated probability that the result answers the question. Relevance still needs evaluation for the application.

Try changing `query_text` to an expense, VPN, or laptop question and inspect how the ranking changes.

## 6. Retrieve by ID and scroll through points

Use `retrieve` when you already know point IDs. This is a direct read, not a similarity search. Use `scroll` when you need to enumerate records (often page by page) or inspect records matching a filter. For large collections, do not load every point into memory at once.

In [7]:
retrieved = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=point_ids[:2],
    with_payload=True,
    with_vectors=False,
)
[(point.id, point.payload["document_id"], point.payload["chunk_index"]) for point in retrieved]

[('86e06dae-8c6e-58ce-963c-a362c436b89b', 'account-access', 0),
 ('9caffaf4-a347-5565-a73f-67a4107cb87d', 'account-access', 1)]

In [8]:
page, next_offset = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=4,
    with_payload=True,
    with_vectors=False,
)
print("Page size:", len(page))
print("Next page offset:", next_offset)
[(point.id, point.payload["document_id"]) for point in page]

Page size: 4
Next page offset: 89218834-fea4-57e9-a83b-caf38761c894


[('26ae8f19-6866-5167-8c39-80cbe94d16e2', 'device-support'),
 ('39b2558d-ed0d-5407-bf5b-abc88d409c6a', 'device-support'),
 ('74e482cd-857b-51fb-a3a7-ffe5d8381d5d', 'expense-policy'),
 ('86e06dae-8c6e-58ce-963c-a362c436b89b', 'account-access')]

Scroll returns a page plus an offset for continuing iteration. The offset is not a search score. In a production export or migration, keep paging until the returned offset is `None`, and consider filters or parallel slices for a large collection.

## 7. Filter search with payload fields

A payload filter constrains vector search to matching records. Our filter is a `must` condition: the result must have `tenant_id == "acme"`. This demonstrates retrieval behavior, not tenant security. Never rely on a user-controlled filter as the only authorization check.

**Local-mode detail:** the Python client's local implementation can evaluate payload filters, but creating payload indexes has no effect there. The following code therefore demonstrates filtering only. To study indexed filtering and its performance, connect to a Qdrant server and create indexes for fields that meaningfully narrow your workload. Indexing every field has a cost, so choose fields deliberately.

For a server-backed client whose collection already exists, the index calls look like this:

```python
server_client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="tenant_id",
    field_schema=models.PayloadSchemaType.KEYWORD,
)
server_client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="category",
    field_schema=models.PayloadSchemaType.KEYWORD,
)
```

The lab's two tenants are for learning filters, not a security boundary. Production authorization needs separate access-control design.

In [9]:
# Local mode evaluates this filter, but does not build payload indexes.
acme_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="tenant_id",
            match=models.MatchValue(value="acme"),
        )
    ]
)

filtered = client.query_points(
    collection_name=COLLECTION_NAME,
    query=models.Document(text=query_text, model=MODEL_NAME),
    query_filter=acme_filter,
    limit=10,
    with_payload=True,
).points

[(hit.payload["tenant_id"], hit.payload["category"], hit.payload["text"]) for hit in filtered]

[('acme',
  'account',
  'If you forget your password, choose Forgot password on the sign-in page and follow the email instructions.'),
 ('acme',
  'account',
  'To sign in, use your work email and the password associated with your account.'),
 ('acme',
  'account',
  'Multi-factor authentication adds a second verification step after you enter your password.'),
 ('acme',
  'it-support',
  'Restart your laptop if it becomes unresponsive. Save your work first when possible.'),
 ('acme',
  'it-support',
  'Connect to the company VPN before opening internal tools from an off-site network.'),
 ('acme',
  'finance',
  'Submit business travel expenses within thirty days and attach itemized receipts.'),
 ('acme',
  'finance',
  'A manager must approve an expense report before reimbursement is issued.')]

You can combine conditions. For example, `must` can require both the tenant and category. Qdrant filter conditions also include `should` and `must_not`; range, text, nested, and other condition types are explored in the later filtering notebook.

In [10]:
finance_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="tenant_id",
            match=models.MatchValue(value="acme"),
        ),
        models.FieldCondition(
            key="category",
            match=models.MatchValue(value="finance"),
        ),
    ]
)

finance_hits = client.query_points(
    collection_name=COLLECTION_NAME,
    query=models.Document(text="Which receipts are needed for reimbursement?", model=MODEL_NAME),
    query_filter=finance_filter,
    limit=3,
    with_payload=True,
).points
[(hit.payload["document_id"], hit.payload["text"]) for hit in finance_hits]

[('expense-policy',
  'A manager must approve an expense report before reimbursement is issued.'),
 ('expense-policy',
  'Submit business travel expenses within thirty days and attach itemized receipts.')]

## 8. CRUD in Qdrant

We have already created points by upserting them. Now we will read and update the same point, then remove it. These operations affect a specific point ID; payload filters can also select sets of points. Qdrant records point changes through its write path, but exact visibility and durability behavior depends on deployment and operation settings.

### Patch a payload field

`set_payload` merges the provided fields into the existing payload. It does not regenerate the vector. Use this for metadata-only changes, such as marking a point reviewed. If the text changes, update the vector too so the stored embedding still represents the stored content.

In [11]:
target_id = point_ids[0]

client.set_payload(
    collection_name=COLLECTION_NAME,
    payload={"reviewed": True, "content_version": 1},
    points=[target_id],
    wait=True,
)

updated = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[target_id],
    with_payload=True,
    with_vectors=False,
)[0]
updated.payload

{'document_id': 'account-access',
 'chunk_index': 0,
 'tenant_id': 'acme',
 'category': 'account',
 'source': 'employee-handbook',
 'text': 'To sign in, use your work email and the password associated with your account.',
 'reviewed': True,
 'content_version': 1}

### Update text and vector together

When the source chunk text changes, embed the new text and upsert the complete point at the same ID. This avoids a mismatch where payload text says one thing but the vector still represents the old text. The example also preserves the other payload fields.

In [ ]:
updated_text = "If you cannot remember your password, select Forgot password and follow the reset link sent to your work email."
old_point = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[point_ids[1]],
    with_payload=True,
    with_vectors=False,
)[0]
updated_payload = {**old_point.payload, "text": updated_text, "content_version": 2}

client.upsert(
    collection_name=COLLECTION_NAME,
    points=[
        models.PointStruct(
            id=point_ids[1],
            vector=models.Document(text=updated_text, model=MODEL_NAME),
            payload=updated_payload,
        )
    ],
    wait=True,
)

client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[point_ids[1]],
    with_payload=True,
    with_vectors=False,
)[0].payload

### Delete one point or all points for a document

Delete by ID when you know the exact point. Delete by a filter when removing all chunks for a source document. Filter-based deletion is powerful: verify the filter and the selected records before running it against important data.

In [ ]:
# Remove one point by its ID.
client.delete(
    collection_name=COLLECTION_NAME,
    points_selector=models.PointIdsList(points=[point_ids[-1]]),
    wait=True,
)

client.count(collection_name=COLLECTION_NAME, exact=True).count

In [ ]:
# Remove all chunks belonging to the travel-policy document.
# This is safe to demonstrate here because the lab collection is disposable.
client.delete(
    collection_name=COLLECTION_NAME,
    points_selector=models.FilterSelector(
        filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="document_id",
                    match=models.MatchValue(value="travel-policy"),
                )
            ]
        )
    ),
    wait=True,
)

client.count(collection_name=COLLECTION_NAME, exact=True).count

## 9. Persistence modes

The main client in this notebook uses **in-memory mode**, so closing it or restarting the kernel discards the collection. For a small local experiment that should survive reopening the client, Qdrant's Python client can use a local directory via `QdrantClient(path=...)`. This is still local embedded storage, not a networked service or a production backup strategy.

The next cell uses a temporary directory and opens the same local database twice. It demonstrates data surviving a client close/reopen while the temporary directory exists. The temporary directory is removed afterward.

In [12]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temporary_directory:
    storage_path = str(Path(temporary_directory) / "qdrant_data")

    local_client = QdrantClient(path=storage_path)
    local_client.create_collection(
        collection_name="persistence_demo",
        vectors_config=models.VectorParams(size=3, distance=models.Distance.COSINE),
    )
    local_client.upsert(
        collection_name="persistence_demo",
        points=[
            models.PointStruct(
                id=1,
                vector=[0.1, 0.2, 0.3],
                payload={"text": "This point is stored in a local directory."},
            )
        ],
        wait=True,
    )
    local_client.close()

    reopened_client = QdrantClient(path=storage_path)
    persisted = reopened_client.retrieve(
        collection_name="persistence_demo",
        ids=[1],
        with_payload=True,
        with_vectors=True,
    )
    print(persisted[0].payload)
    reopened_client.close()

{'text': 'This point is stored in a local directory.'}


To keep a local database between notebook runs, use a stable path such as `./qdrant_local_data` instead of `TemporaryDirectory`. Close the client before opening the same local path again. Do not commit local database data to source control unless that is intentional.

For a Qdrant server, connect with `QdrantClient(url="http://localhost:6333")`. A server deployment has its own storage configuration and operational lifecycle; the official quickstart demonstrates running it with Docker and mounting a storage volume. We will return to server deployment, backup/restore, and access control in the production architecture notebook.

## 10. Inspect and clean up

For this lab, inspect collection counts and the client state before cleanup. The final cleanup closes the in-memory client. If you used a stable local path instead, closing the client preserves its data on disk.

In [13]:
print("Final lab point count:", client.count(collection_name=COLLECTION_NAME, exact=True).count)
client.close()

Final lab point count: 9


## Troubleshooting guide

- **`ImportError` for FastEmbed support:** install `qdrant-client[fastembed]` in the active kernel environment and restart the kernel.
- **First run is slow:** FastEmbed may be downloading/loading the embedding model.
- **Dimension mismatch:** create the collection with the embedding model's output size and use the same model for document and query vectors.
- **Collection already exists:** rerun the reset/create cell for this disposable in-memory lab, or choose a new collection name.
- **Local path is locked:** close any Qdrant client using that path before opening it again.
- **Expected text is missing from hits:** request payloads and make sure the text is stored in the payload.
- **Filter returns no hits:** inspect the payload values and check that field names and types match the filter.
- **A changed passage ranks oddly:** regenerate and upsert its vector together with its new text payload.

## Exercises

1. Change the query to ask about submitting a business expense, then print the top three document IDs and scores.
2. Add a `language` payload field and filter for one language. What changes in the result set?
3. Use `retrieve` to compare the point before and after `set_payload`. Which fields changed, and did the vector change?
4. Update one chunk's text and vector, then query both the old and new phrasing. What do you observe?
5. Re-ingest the same document using the same point IDs. What behavior makes this safe to retry? What cleanup is needed if its number of chunks shrinks?
6. Run the persistence example with a stable local directory and reopen the client in a later kernel session. What remains, and what would still be missing from a complete production recovery plan?

## Summary

You created a Qdrant collection, embedded and upserted help-center chunks, ran semantic top-K queries, retrieved points by ID, paged through records, filtered on payload fields, updated payload and vector data, and deleted points. You also compared in-memory and local persistent modes.

The key operational habit is to keep each text chunk, its vector, its stable ID, and its payload in sync. The next notebook compares Qdrant with FAISS and pgvector; later notebooks go deeper into filtering, performance, and production architecture.